# Neural Networks Playground: Classification and Regression

## 1. Introduction
This Jupyter notebook is designed to work in Google Colab so you can experiment with small neural networks using Keras/TensorFlow.

- The code is in python which is aligned with the Colab environment and the keras and tensorflow capabilities
- **You do not need to know how to code in Python to experiment with the models.**

You will compare:

- baseline statistical / machine learning models
- small neural networks
- different network architectures
- different activation functions
- learning rates
- batch sizes
- epochs
- regularization and dropout

The goal is not to build the biggest model. The goal is to understand how model configuration affects training, overfitting, and test-set performance.

### 1.1 Learning Outcomes

By the end, you should be able to explain:

1. What a forward pass computes
2. What the loss function measures
3. What changes during training
4. Why more layers or more neurons do not always improve results
5. Why baseline models can outperform neural networks on some tabular datasets

### 1.3 Runtime notes

Go to Google Colab <https://colab.research.google.com/> and upload this file.

This notebook should run on the free CPU runtime. You do **not** need a GPU for this notebook.

Recommended: Check under `Connect` that  `Runtime` is set to `CPU`

Then run the cells from top to bottom.

- Click on the right facing arrow at the top left of the cell.

## 2. Setup: Import libraries and set random seeds

This cell loads all the Python packages needed for the notebook.

- **numpy** and **pandas** handle numerical arrays and data frames.
- **matplotlib** produces the training-history plots.
- **scikit-learn** supplies the datasets, train/test splitting, feature scaling, baseline models, and evaluation metrics.
- **tensorflow / keras** build and train the neural networks.

Setting `np.random.seed` and `tf.random.set_seed` to the same value makes results as reproducible as possible across runs. Note that exact reproducibility is not guaranteed on GPU hardware.

The final `print` statement confirms which version of TensorFlow is active — useful for debugging if something behaves unexpectedly.

In [ ]:
# Core imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

np.random.seed(427)
tf.random.set_seed(427)

print("TensorFlow version:", tf.__version__)

## 3. Create Helper functions

These three functions are used throughout the notebook so we do not repeat boilerplate code.

1. `plot_history(history, metric, title)`

 Plots the **loss** curve (and optionally a second metric such as accuracy or MAE) for both the training set and the validation set across epochs. Watching these two curves together is the primary tool for diagnosing overfitting:

- If training loss keeps falling but validation loss levels off or rises, the model is memorising the training data rather than generalising.
- If both curves fall together and are close to each other, training is healthy.

2. `build_classification_model(input_dim, hidden_layers, activation, learning_rate, dropout_rate, l2_penalty)`
Constructs a **binary classification** neural network using Keras's Sequential API.

- The `hidden_layers` argument is a Python list where each integer specifies the number of neurons in one hidden layer. For example, `[32, 16]` creates two hidden layers with 32 and 16 neurons respectively.
- Each hidden layer uses the chosen `activation` function (default: ReLU) and optionally applies L2 weight regularization.
- A `Dropout` layer can be inserted after each hidden layer to randomly zero a fraction of activations during training; this is a regularisation technique that reduces overfitting.
- The **output layer** has a single neuron with a **sigmoid** activation, which squashes the output to the range [0, 1] so it can be interpreted as a probability of belonging to the positive class.
- The model is compiled with **binary cross-entropy** loss (appropriate for binary classification) and the **Adam** optimiser.

3. `build_regression_model(input_dim, hidden_layers, activation, learning_rate, dropout_rate, l2_penalty)`
Identical structure to the classification model, except:

- The output layer has **no activation function** (a linear output), so predictions are unbounded real numbers which is appropriate for regression.
- The loss function is **mean squared error (MSE)**, and the reported metric is **mean absolute error (MAE)**.

In [ ]:
def plot_history(history, metric=None, title="Training history"):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="training loss")
    if "val_loss" in hist:
        plt.plot(hist["val_loss"], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " -- Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    if metric is not None and metric in hist:
        plt.figure(figsize=(8, 5))
        plt.plot(hist[metric], label=f"training {metric}")
        val_metric = "val_" + metric
        if val_metric in hist:
            plt.plot(hist[val_metric], label=f"validation {metric}")
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.title(title + f" -- {metric}")
        plt.legend()
        plt.grid(True)
        plt.show()


def build_classification_model(input_dim, hidden_layers=[16], activation="relu", learning_rate=0.001, dropout_rate=0.0, l2_penalty=0.0):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation, kernel_regularizer=regularizers.l2(l2_penalty)))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_regression_model(input_dim, hidden_layers=[16], activation="relu", learning_rate=0.001, dropout_rate=0.0, l2_penalty=0.0):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation, kernel_regularizer=regularizers.l2(l2_penalty)))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
    return model

## 4. Classification Example

We will use the breast cancer dataset from scikit-learn.

The task is binary classification. The predictors are numeric measurements from medical images.

This is a small tabular dataset, which is exactly the kind of setting where a simpler baseline model may perform very well.

### 4.1 Load the classification dataset

The **Breast Cancer Wisconsin** dataset (IC Irvine Machine Learning Repository <https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic>)  contains measurements computed from digitized images of fine needle aspirates (FNA) of breast masses. 

Each row represents one tumor, described by 30 numerical features (e.g., radius, texture, smoothness of cell nuclei). The binary target is:

- `0` = malignant
- `1` = benign

Calling `X_class.head()` at the end of this cell displays the first five rows so you can see the feature names and data types. The dataset has 569 samples, small enough that a logistic regression baseline may already achieve near-ceiling accuracy.

In [ ]:
# Load classification data

cancer = load_breast_cancer()
X_class = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_class = cancer.target

print("Shape:", X_class.shape)
print("Target names:", cancer.target_names)
X_class.head()

### 4.2 Train/test split and feature scaling

Two steps happen here that are essential for any ML workflow.

**1. Split the data into Train and Test Sets** using train_test_split

We hold out 25 % of the data as a test set. The stratify_class argument ensures the proportion of malignant vs. benign cases is the same in both splits which is important when classes are imbalanced.

**2. Scale the Features (variables)** using StandardScaler

Neural networks (and many other gradient-based models) are sensitive to the scale of their inputs. `StandardScaler` subtracts the mean and divides by the standard deviation for each feature, so all inputs have mean ≈ 0 and variance ≈ 1 after scaling.

> **Important**: The scaler is *fitted* only on the training set (using fit_transform()), then applied to the test set (`transform`). Fitting on the test set would be **data leakage** since the model would have indirect access to test-set statistics during training.

In [ ]:
# Train/test split and scaling

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.25, random_state=427, stratify=y_class)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print("Training rows:", X_train_c_scaled.shape[0])
print("Testing rows:", X_test_c_scaled.shape[0])

### 4.3 Classification baseline: logistic regression

Before using a neural network, fit a simple baseline model. This gives us a benchmark.

Logistic regression is the classical method for binary classification. It fits a linear decision boundary in the feature space. For well-separated, relatively small tabular datasets like Breast Cancer, it often achieves very high accuracy with minimal tuning.

The function make_pipeline(StandardScaler(), LogisticRegression(...)) ensures scaling happens inside the pipeline, so we pass the *un-scaled* training data here (the pipeline handles it internally).

The **classification report** shows precision, recall, and F1 for each class:

- **Precision**: of all cases predicted positive, how many truly were?
- **Recall**: of all truly positive cases, how many did we catch?
- **F1**: harmonic mean of precision and recall — useful when classes are imbalanced.

> **What to expect**: Logistic regression typically achieves ~95–97% accuracy on this dataset. Keep this number in mind when you evaluate the neural network below.

In [ ]:
baseline_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline_clf.fit(X_train_c, y_train_c)
baseline_pred_c = baseline_clf.predict(X_test_c)

print("Baseline logistic regression accuracy:")
print(accuracy_score(y_test_c, baseline_pred_c))

print("\nClassification report:")
print(classification_report(y_test_c, baseline_pred_c, target_names=cancer.target_names))

### 4.4 Define the Classification neural network by setting hyperparameters

The cell below is the only cell you need to edit when running experiments. Change the values, then re-run this cell and the two cells that follow.

### What each hyperparameter controls

| Hyperparameter | What it controls | Effect of increasing |
|---|---|---|
| `hidden_layers` | Number of layers and neurons per layer | More capacity; may overfit small datasets |
| `activation` | Non-linearity at each hidden neuron | ReLU is fast; tanh saturates; sigmoid is slow |
| `learning_rate` | Step size for gradient descent | Too high → unstable; too low → slow |
| `epochs` | Number of full passes through training data | Too many → overfitting |
| `batch_size` | Samples used per gradient update | Larger = smoother but slower per epoch |
| `dropout_rate` | Fraction of neurons randomly zeroed per step | Reduces overfitting; too high → underfitting |
| `l2_penalty` | L2 weight regularisation strength | Shrinks weights; reduces overfitting |

### Suggested experiments (try in order)

1. Start with `[16]` — one hidden layer with 16 neurons
2. Try `[32, 16]` — two hidden layers
3. Try `[128, 64, 32]` — three larger hidden layers
4. Try a higher learning rate such as `0.01`
5. Try too many epochs (e.g., 200) and watch for overfitting in the plots
6. Add dropout (`0.2` or `0.5`) or L2 (`0.001`) and see if it helps

In [ ]:
# Classification hyperparameters

classification_hidden_layers = [16]      # Try [8], [16], [32, 16], [128, 64, 32]
classification_activation = "relu"       # Try "relu", "tanh", "sigmoid"
classification_learning_rate = 0.001     # Try 0.01, 0.001, 0.0001
classification_epochs = 30               # Try 10, 30, 100
classification_batch_size = 16           # Try 8, 16, 32, 64
classification_dropout_rate = 0.0        # Try 0.0, 0.2, 0.5
classification_l2_penalty = 0.0          # Try 0.0, 0.001, 0.01

### 4.5 Build, train, and plot the classification neural network

This cell:

1. Calls `build_classification_model()` with the hyperparameters you set above and prints a **model summary** showing the number of parameters in each layer.
2. Calls `model.fit()` to train the network. `validation_split=0.2` holds out 20% of the training data as **an internal validation set**, so you can monitor generalization during training without touching the test set.
3. Calls `plot_history()` to draw the loss and accuracy curves.

> **What to look for in the plots**:
> - Both curves declining together suggests healthy training
> - Validation loss rising while training loss falls suggests overfitting; try fewer epochs, more dropout, or L2
> - Both curves noisy suggests try a smaller learning rate or larger batch size

In [ ]:
classification_model = build_classification_model(input_dim=X_train_c_scaled.shape[1], hidden_layers=classification_hidden_layers, activation=classification_activation, learning_rate=classification_learning_rate, dropout_rate=classification_dropout_rate, l2_penalty=classification_l2_penalty)

classification_model.summary()

history_c = classification_model.fit(X_train_c_scaled, y_train_c, validation_split=0.2, epochs=classification_epochs, batch_size=classification_batch_size, verbose=0)

plot_history(history_c, metric="accuracy", title="Classification neural network")

### 4.6 Evaluate the classification neural network on the test set

Now we measure the model's performance on data it has never seen during training or validation.

- `model.evaluate()` returns the loss and accuracy on the test set.
- `model.predict()` returns the **sigmoid output** — a probability between 0 and 1 for each observation. We convert this to a class label using a threshold of 0.5: predicted class = 1 (benign) if probability ≥ 0.5, else 0 (malignant).
- The **confusion matrix** shows counts in a 2×2 table: true negatives, false positives, false negatives, true positives.

> **Compare the neural network accuracy to the logistic regression accuracy from the baseline step.** On this dataset, it is common to find that logistic regression matches or outperforms a small neural network, especially with few epochs.

In [ ]:
test_loss_c, test_acc_c = classification_model.evaluate(X_test_c_scaled, y_test_c, verbose=0)
pred_prob_c = classification_model.predict(X_test_c_scaled, verbose=0).ravel()
pred_class_c = (pred_prob_c >= 0.5).astype(int)

print("Neural network test accuracy:", test_acc_c)
print("Neural network test loss:", test_loss_c)

print("\nConfusion matrix:")
print(confusion_matrix(y_test_c, pred_class_c))

print("\nClassification report:")
print(classification_report(y_test_c, pred_class_c, target_names=cancer.target_names))

### 4.7 Classification reflection questions with sample answers

Answer these questions after trying several configurations.

**1. Did the neural network beat logistic regression?**

> **Sample answer**: Usually not, or only marginally. With 569 observations and 30 numerical features, logistic regression reaches ~96–97% accuracy with no tuning. A small neural network trained for 30 epochs achieves similar or slightly lower accuracy. This illustrates a key principle: on small, well-structured tabular datasets, simpler models are hard to beat because they have fewer parameters to overfit and the decision boundary is approximately linear.

**2. Did adding more layers always improve test accuracy?**

> **Sample answer**: No. Going from `[16]` to `[32, 16]` may give a small gain, but `[128, 64, 32]` often *hurts* on this dataset. A 3-layer network with 128 + 64 + 32 = 224 neurons has thousands of parameters to learn from only ~426 training samples. Without strong regularization it memorizes the training data and generalizses poorly, the classic indicator of overfitting to the noise in the training data.

**3. Did training accuracy and validation accuracy move together?**

> **Sample answer**: For small, well-regularized models (e.g., `[16]`, 30 epochs) both curves improve together and remain close. For large models or too many epochs the gap widens: training accuracy continues to rise while validation accuracy plateaus or falls. A large and persistent gap between the two curves is the primary diagnostic signal for overfitting.

**4. Did you see signs of overfitting?**

> **Sample answer**: Yes, when using a large network (e.g., `[128, 64, 32]`) or many epochs (e.g., 200), the training loss continues to decrease while the validation loss starts to increase. The model begins to memorize noise in the training samples rather than learning the underlying pattern. Adding `dropout_rate=0.2` or `l2_penalty=0.001` typically reduces this gap.

**5. Which hyperparameter seemed to matter most?**

> **Sample answer**: The **number of epochs** and **network size** (hidden_layers) have the largest effect on whether overfitting occurs. The **learning rate** has the largest effect on whether training converges at all; too high causes the loss to oscillate or diverge, too low causes very slow learning. Dropout and L2 are secondary but important once overfitting is detected.

## 4b. Image Classification Example — Fashion-MNIST

The breast cancer example used a **tabular data**: each row is a structured record with named features.

Neural networks have a more decisive advantage when the input is **unstructured data** such as images, where the network must 
discover useful features (edges, shapes, textures) rather than being given them.

This section introduces a second classification example using the **Fashion-MNIST** dataset <https://github.com/zalandoresearch/fashion-mnist>:

- 70,000 grayscale images of clothing items, each 28 × 28 pixels
- 10 classes: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- A standard benchmark in computer vision, introduced by Zalando Research (2017) as a harder 
replacement for the original handwritten-digit MNIST

We will compare:

1. A **logistic regression baseline** applied to the flattened 784-pixel vectors
2. A **dense (fully connected) neural network** — the same architecture we used in Part 4

> **Key learning goal**: see how the same neural network building blocks scale to image data, 
and why even a simple dense network outperforms logistic regression on image classification.

*Note: Convolutional neural networks (CNNs) would do even better but this notebook uses only 
dense layers to keep the the NN architecture consistent across examples.*

### 4b.1 Load Fashion-MNIST and display a sample image grid

Fashion-MNIST is bundled directly inside Keras, so no download is required. 

- `keras.datasets.fashion_mnist.load_data()` returns pre-split train and test arrays.

The raw pixel values are integers in [0, 255]. We divide by 255 to rescale them to [0, 1], 
which is the standard pre-processing step for image neural networks.

The grid below displays a **5 × 10** sample: 5 randomly chosen examples per class — 
so you can see what the network must learn to distinguish.

Notice:

- Images within the same class have significant variation in pose, texture and colour tone
- Some classes look very similar (Shirt vs. T-shirt/top; Sneaker vs. Ankle boot)
- There is no pre-engineered feature — the only input to the model is the raw pixel matrix

In [ ]:
# Load Fashion-MNIST from Keras (no separate download needed)

fashion_class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",      "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

(X_f_train_raw, y_f_train), (X_f_test_raw, y_f_test) = keras.datasets.fashion_mnist.load_data()

# Normalise pixel values from [0, 255] to [0.0, 1.0]
X_f_train = X_f_train_raw / 255.0
X_f_test  = X_f_test_raw  / 255.0

print(f"Training images : {X_f_train.shape}   (samples, height, width)")
print(f"Test images     : {X_f_test.shape}")
print(f"Training labels : {y_f_train.shape},  unique classes: {len(set(y_f_train))}")

: 

In [ ]:
# Display a 5 x 10 sample grid — 5 examples of each of the 10 classes

n_per_class = 5
n_classes   = 10

fig, axes = plt.subplots(n_per_class, n_classes, figsize=(14, 7))
fig.suptitle("Fashion-MNIST: sample images (5 per class)", fontsize=14, y=1.01)

for cls in range(n_classes):
    idx = np.where(y_f_train == cls)[0]
    chosen = np.random.choice(idx, size=n_per_class, replace=False)
    for row, img_idx in enumerate(chosen):
        ax = axes[row, cls]
        ax.imshow(X_f_train[img_idx], cmap="gray_r")
        ax.axis('off')
        if row == 0:
            ax.set_title(fashion_class_names[cls], fontsize=8, pad=4)

plt.tight_layout()
plt.show()

### 4b.2 Flatten images for a dense (fully connected) network

A dense neural network expects a **1-D feature vector** as input, not a 2-D image matrix. 

- We reshape each 28 × 28 image into a flat vector of 784 numbers.

This is called **flattening** and is the simplest way to feed image pixels into the same kind 
of network we built in Part 4. The trade-off is that spatial relationships between neighboring 
pixels are lost; pixel (3, 4) and pixel (3, 5) are just two independent features as far as 
the dense layer is concerned.

A **convolutional network** preserves this spatial structure; that is why CNNs dominate image 
benchmarks. But for this exercise, flattening lets us reuse `build_classification_model()` directly.

> **Input dimensionality**: 28 × 28 = **784 features**, 26 times more than the breast cancer 
dataset. This means more parameters per layer and longer training times, but also a richer input representation.

In [ ]:
# Flatten 28x28 images into 784-element vectors

X_f_train_flat = X_f_train.reshape(len(X_f_train), -1)   # shape: (60000, 784)
X_f_test_flat  = X_f_test.reshape(len(X_f_test),  -1)    # shape: (10000, 784)

print("Flattened training shape:", X_f_train_flat.shape)
print("Flattened test shape    :", X_f_test_flat.shape)

### 4b.3 Baseline: multi-class logistic regression on flattened pixels

We fit a **multi-class logistic regression** (softmax regression) on the flattened pixel vectors.

- `max_iter=500` — more iterations are needed because the input has 784 features
- `C=0.1` — light L2 regularisation (C is the *inverse* of regularisation strength; 
smaller C = stronger penalty) to help convergence
- `solver='saga'` — an efficient solver for large datasets with L1/L2 penalties
- `n_jobs=-1` — use all available CPU cores

On Fashion-MNIST, logistic regression typically achieves **~84% accuracy**. For reference:

| Model | Typical accuracy |
|---|---|
| Random guessing (10 classes) | 10% |
| Logistic regression on pixels | ~84% |
| Dense NN (this notebook) | ~88–90% |
| CNN (not in this notebook) | >99% |

Because pixels are already in [0, 1], we do **not** use `StandardScaler` here.

In [ ]:
# Baseline: multi-class logistic regression on flattened pixels
# Pixels are already in [0,1] so no additional scaling needed

from sklearn.linear_model import LogisticRegression as _LR  # avoid re-import conflict

print("Fitting logistic regression baseline (may take ~30–60 seconds on CPU)...")
baseline_fashion = _LR(max_iter=500, C=0.1, solver='saga', n_jobs=-1)
baseline_fashion.fit(X_f_train_flat, y_f_train)

baseline_fashion_pred = baseline_fashion.predict(X_f_test_flat)
baseline_fashion_acc  = accuracy_score(y_f_test, baseline_fashion_pred)

print(f"\nBaseline logistic regression accuracy: {baseline_fashion_acc:.4f}")
print("\nClassification report:")
print(classification_report(y_f_test, baseline_fashion_pred,
                            target_names=fashion_class_names))

### 4b.4 Build a multi-class dense neural network for Fashion-MNIST

Fashion-MNIST has **10 classes**, so the output layer must change from the binary classifier:

| Setting | Binary (Part 4) | Multi-class (Part 4b) |
|---|---|---|
| Output neurons | 1 | 10 (one per class) |
| Output activation | sigmoid | softmax |
| Loss function | binary crossentropy | sparse categorical crossentropy |
| Prediction | threshold at 0.5 | argmax of 10 probabilities |

**Softmax** converts raw output scores (logits) into a probability distribution summing to 1 across all 10 classes.

**Sparse categorical crossentropy** expects integer class labels (0–9) rather than one-hot vectors, 
which is convenient because Fashion-MNIST labels are already integers.

The `build_fashion_model()` function below is a new helper tailored for 10-class image classification. 
The hidden layer architecture is otherwise identical to `build_classification_model()`.

In [ ]:
def build_fashion_model(hidden_layers=[128, 64], activation='relu',
                        learning_rate=0.001, dropout_rate=0.0, l2_penalty=0.0):
    """Dense neural network for 10-class Fashion-MNIST classification."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(784,)))          # 28*28 flattened pixels

    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation,
                               kernel_regularizer=regularizers.l2(l2_penalty)))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(10, activation='softmax'))  # 10-class output

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


# Print summary for default architecture
build_fashion_model().summary()

### 4b.5 Set hyperparameters for the Fashion-MNIST network

Because the input has 784 features, we start with larger hidden layers than Part 4.

| Experiment | Setting | Expected effect |
|---|---|---|
| Small network | `[64]` | Underfits — accuracy stays ~85% |
| Default | `[128, 64]` | Good balance — ~87–89% |
| Wider | `[256, 128]` | Marginal gain, more compute |
| Add dropout | `dropout_rate=0.3` | Reduces overfitting gap |
| More epochs | 15 → 25 | Accuracy keeps improving slowly |
| High learning rate | `0.01` | Loss may oscillate |

> **Note**: With 60,000 training images each epoch takes longer than Parts 4 and 5. 
Keep `epochs ≤ 20` for interactive exploration, or increase to 30 if using a Colab GPU runtime 
(`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Fashion-MNIST hyperparameters — edit these values to experiment

fashion_hidden_layers  = [128, 64]    # Try [64], [128, 64], [256, 128]
fashion_activation     = "relu"       # Try "relu", "tanh"
fashion_learning_rate  = 0.001        # Try 0.01, 0.001, 0.0001
fashion_epochs         = 15           # Try 10, 15, 20, 25
fashion_batch_size     = 64           # Try 32, 64, 128
fashion_dropout_rate   = 0.2          # Try 0.0, 0.2, 0.3, 0.5
fashion_l2_penalty     = 0.0          # Try 0.0, 0.0001, 0.001

### 4b.6 Build, train, and plot the Fashion-MNIST network

Same workflow as Part 4: build → summary → fit with validation split → plot curves.

**What to look for in the plots**:

- Training accuracy should rise steadily across epochs
- Validation accuracy typically lags a few points — a small gap is normal; 
a rapidly growing gap signals overfitting
- With `dropout_rate=0.2` the gap should be smaller than with `0.0`

> **On a Colab CPU**: expect ~20–40 seconds per epoch. Switching to a T4 GPU reduces this to ~3–5 seconds.

In [ ]:
tf.random.set_seed(427)

fashion_model = build_fashion_model(
    hidden_layers  = fashion_hidden_layers,
    activation     = fashion_activation,
    learning_rate  = fashion_learning_rate,
    dropout_rate   = fashion_dropout_rate,
    l2_penalty     = fashion_l2_penalty
)

fashion_model.summary()

history_f = fashion_model.fit(
    X_f_train_flat, y_f_train,
    validation_split = 0.2,
    epochs           = fashion_epochs,
    batch_size       = fashion_batch_size,
    verbose          = 1
)

plot_history(history_f, metric="accuracy", title="Fashion-MNIST dense network")

### 4b.7 Evaluate on the test set and visualise predictions

We evaluate the trained network on the held-out 10,000 test images and then plot a grid 
of 25 random test images with their true and predicted labels.

- **Green title** = correct prediction
- **Red title** = wrong prediction (includes confidence %)

The per-class breakdown in the classification report reveals which garments are hardest:

- Visually distinctive classes (Trouser, Bag, Ankle boot) tend to have high recall
- Visually similar classes (Shirt vs. T-shirt/top vs. Pullover vs. Coat) are the main sources of confusion

Look at the misclassified images — can *you* tell them apart?

In [ ]:
# Evaluate on test set
test_loss_f, test_acc_f = fashion_model.evaluate(X_f_test_flat, y_f_test, verbose=0)
pred_probs_f   = fashion_model.predict(X_f_test_flat, verbose=0)  # (10000, 10)
pred_classes_f = np.argmax(pred_probs_f, axis=1)                   # argmax over 10 probs

print(f"Fashion-MNIST neural network test accuracy : {test_acc_f:.4f}")
print(f"Baseline logistic regression accuracy     : {baseline_fashion_acc:.4f}")
print()
print("Classification report (neural network):")
print(classification_report(y_f_test, pred_classes_f, target_names=fashion_class_names))

In [ ]:
# Visualise 25 random test predictions

n_show     = 25
sample_idx = np.random.choice(len(X_f_test), size=n_show, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(11, 11))
fig.suptitle("Fashion-MNIST: test predictions (green = correct, red = wrong)",
             fontsize=13, y=1.01)

for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_f_test[idx], cmap="gray_r")
    ax.axis('off')
    true_label  = fashion_class_names[y_f_test[idx]]
    pred_label  = fashion_class_names[pred_classes_f[idx]]
    confidence  = pred_probs_f[idx, pred_classes_f[idx]]
    colour      = 'green' if y_f_test[idx] == pred_classes_f[idx] else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.0%})",
                 color=colour, fontsize=7)

plt.tight_layout()
plt.show()

### 4b.8 Confusion matrix

A 10 × 10 confusion matrix shows where the model's errors cluster. 
Rows are true labels; columns are predicted labels. Diagonal cells are correct predictions.

Look for **off-diagonal concentrations** — these reveal which classes are frequently confused. 
For Fashion-MNIST the most common confusions are:

- Shirt ↔ T-shirt/top
- Shirt ↔ Pullover
- Coat ↔ Pullover
- Sneaker ↔ Ankle boot

These are visually similar pairs. A CNN with spatial feature learning substantially reduces these confusions.

In [ ]:
cm_f = confusion_matrix(y_f_test, pred_classes_f)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_f, cmap="Blues")
plt.colorbar(im, ax=ax)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(fashion_class_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(fashion_class_names, fontsize=9)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion matrix — Fashion-MNIST neural network")

# Annotate each cell with the count
thresh = cm_f.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm_f[i, j]),
                ha='center', va='center',
                color='white' if cm_f[i, j] > thresh else 'black',
                fontsize=8)

plt.tight_layout()
plt.show()

### 4b.9 Fashion-MNIST reflection questions with sample answers

Answer these questions after running the notebook with several configurations.

**1. Did the neural network outperform logistic regression? By how much?**

> **Sample answer**: Yes, clearly. Logistic regression typically reaches ~84% accuracy on Fashion-MNIST. 
The dense neural network with `[128, 64]` hidden layers and 15 epochs achieves ~88–89%. 
That 4–5 percentage-point gap represents 400–500 additional correct classifications on the 10,000-sample test set. 
The gap is much larger than what we observed for breast cancer (where logistic regression was nearly unbeatable). 
The reason: image pixels have non-linear spatially-structured relationships that a linear model cannot capture, 
but hidden dense layers partially can.

**2. Which garment classes had the lowest accuracy and why?**

> **Sample answer**: Shirt, T-shirt/top, and Pullover consistently have the lowest per-class recall (often 70–80%). 
They share similar silhouettes: a body with two sleeves, roughly the same aspect ratio, and similar textures. 
Without spatial feature detectors (which CNNs provide), the dense network cannot reliably distinguish them from 
raw pixel values. In contrast, Trouser, Bag, and Ankle boot have distinctive shapes and achieve recall above 95%.

**3. What does the confusion matrix tell you that the overall accuracy number does not?**

> **Sample answer**: Overall accuracy collapses model quality into a single number but hides *which* specific errors 
the model makes. The confusion matrix reveals that errors cluster in particular class pairs (Shirt ↔ T-shirt/top, 
Coat ↔ Pullover). This is actionable: a practitioner could collect more training data for confused classes, 
engineer class-specific features, or switch to a CNN. Two models with identical overall accuracy can have very 
different error patterns — the confusion matrix makes that visible.

**4. Did dropout reduce the gap between training and validation accuracy?**

> **Sample answer**: Yes. Without dropout (`dropout_rate=0.0`), training accuracy often exceeds validation accuracy 
by 3–5 points after 15 epochs, and the gap tends to widen. With `dropout_rate=0.2` the gap narrows to 1–2 points 
and the validation curve is smoother. The trade-off is that training accuracy itself is lower (neurons are randomly 
zeroed during training), but test-set accuracy — the metric that actually matters — is typically higher or equal.

**5. How does this example change your thinking about when to use a neural network?**

> **Sample answer**: The breast cancer example showed that neural networks do not automatically beat simpler models 
on small, well-structured tabular data. Fashion-MNIST shows that neural networks have a genuine edge when:
>
> - The input is **high-dimensional** (784 pixels vs. 30 medical measurements)
> - The features are **raw and unstructured** (pixels without domain-engineered meaning)
> - The dataset is **large enough** (60,000 training images vs. 426 training cases)
>
> Even so, the dense network is not state of the art — a CNN would add another ~10 percentage points. 
This suggests a practical hierarchy: for tabular data, start with logistic regression or gradient boosting; 
for image data, start with a CNN; for text, start with a transformer. Dense networks are a useful pedagogical tool, 
but rarely the best choice for production image classification.

##  5. Regression Example

We will use the California housing dataset (< https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset>).

The task is regression: predicting median house value from neighborhood-level features.

This is a tabular prediction problem. Linear models often provide a strong baseline.

### 5.1 Load the regression dataset

The **California Housing** dataset is derived from the 1990 U.S. Census. Each row represents one census block group (a small geographic unit), described by 8 features:

| Feature | Description |
|---|---|
| `MedInc` | Median income in block group (in tens of thousands of dollars) |
| `HouseAge` | Median house age in block group |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block group population |
| `AveOccup` | Average number of household members |
| `Latitude` | Block group latitude |
| `Longitude` | Block group longitude |

The **target** (`y_reg`) is the **median house value** in hundreds of thousands of dollars (so a value of `2.0` means $200,000). The dataset has ~20,000 rows — larger than the classification dataset, which gives the neural network more data to learn from.

In [2]:
housing = fetch_california_housing(as_frame=True)
X_reg = housing.data
y_reg = housing.target

print("Shape:", X_reg.shape)
X_reg.head()

NameError: name 'fetch_california_housing' is not defined

### 5.2 Train/test split and feature scaling for regression

The same two-step workflow as Part 1:

1. **Split**: 75% training, 25% test, fixed random state for reproducibility. Note that `stratify` is not used here because the target is continuous.
2. **Scale**: Fit `StandardScaler` on training data only, then apply to both sets.

With ~15,000 training rows, neural networks have substantially more data here than in the classification task, which improves their chance of learning useful representations.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.25, random_state=427)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)

print("Training rows:", X_train_r_scaled.shape[0])
print("Testing rows:", X_test_r_scaled.shape[0])

### 5.3 Regression baseline: Ridge Regression

Fit a simple regularized linear model first. This gives the neural network a meaningful benchmark.

**Ridge regression** is ordinary least squares with an L2 penalty added to the loss, which shrinks coefficients and reduces overfitting. The `alpha=1.0` parameter controls the strength of this penalty.

We report three metrics:

| Metric | Interpretation |
|---|---|
| **RMSE** (root mean squared error) | Average prediction error in the same units as the target. Penalises large errors more. |
| **MAE** (mean absolute error) | Average absolute prediction error. More robust to outliers than RMSE. |
| **R²** (coefficient of determination) | Proportion of variance in the target explained by the model. 1.0 = perfect; 0.0 = no better than predicting the mean. |

> **What to expect**: Ridge regression typically achieves RMSE ≈ 0.72–0.75 and R² ≈ 0.60–0.62 on this dataset. This is a meaningful but imperfect fit since house values are driven by non-linear geography effects (latitude/longitude) that a linear model cannot capture.

In [ ]:
baseline_reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
baseline_reg.fit(X_train_r, y_train_r)
baseline_pred_r = baseline_reg.predict(X_test_r)

baseline_rmse = np.sqrt(mean_squared_error(y_test_r, baseline_pred_r))
baseline_mae = mean_absolute_error(y_test_r, baseline_pred_r)
baseline_r2 = r2_score(y_test_r, baseline_pred_r)

print("Baseline ridge regression:")
print(f"RMSE: {baseline_rmse:.3f}")
print(f"MAE:  {baseline_mae:.3f}")
print(f"R^2:  {baseline_r2:.3f}")

### 5.4 Define the Regression neural network by setting hyperparameters

Edit the values below, then re-run this cell and the two that follow.

### Suggested experiments (try in order)

1. Start with `[16]` — compare to ridge regression
2. Try `[64, 32]` — does a wider/deeper network help?
3. Try `[128, 64, 32]` — more capacity
4. Try a higher learning rate (e.g., `0.01`) — does convergence speed up?
5. Try dropout (`0.2`) — does it reduce the train/validation gap?
6. Try L2 regularisation (`0.001`) — compare to dropout
7. Try more epochs (`100`) with a small network — when does it stop improving?

> **Note**: Because this dataset is larger, a neural network has a better chance of beating the linear baseline compared to the breast cancer example in Part 1.

In [ ]:
# Regression hyperparameters

regression_hidden_layers = [16]       # Try [8], [16], [64, 32], [128, 64, 32]
regression_activation = "relu"        # Try "relu", "tanh", "sigmoid"
regression_learning_rate = 0.001      # Try 0.01, 0.001, 0.0001
regression_epochs = 30                # Try 10, 30, 100
regression_batch_size = 32            # Try 16, 32, 64, 128
regression_dropout_rate = 0.0         # Try 0.0, 0.2, 0.5
regression_l2_penalty = 0.0           # Try 0.0, 0.001, 0.01

### 5.5 Build, train, and plot the regression neural network

Identical workflow to the classification model, but using `build_regression_model()`.

The **loss** here is MSE (mean squared error) and the tracked metric is **MAE** (mean absolute error) — both are in units of $100,000s (the target's scale before we scaled the predictors).

Look at the validation MAE curve: if it flattens while training MAE keeps falling, more epochs are not adding value and may be hurting generalisation.

In [ ]:
regression_model = build_regression_model(input_dim=X_train_r_scaled.shape[1], hidden_layers=regression_hidden_layers, activation=regression_activation, learning_rate=regression_learning_rate, dropout_rate=regression_dropout_rate, l2_penalty=regression_l2_penalty)

regression_model.summary()

history_r = regression_model.fit(X_train_r_scaled, y_train_r, validation_split=0.2, epochs=regression_epochs, batch_size=regression_batch_size, verbose=0)

plot_history(history_r, metric="mae", title="Regression neural network")

### 5.6 Evaluate the regression neural network on the test set

Compute RMSE, MAE, and R² for the neural network and print them alongside the baseline ridge regression results for direct comparison.

> **What to expect with defaults** (`[16]`, 30 epochs):
> - NN RMSE ≈ 0.73–0.80 — roughly similar to ridge regression
> - With `[64, 32]`, 100 epochs, the NN typically reaches RMSE ≈ 0.55–0.65 and R² ≈ 0.70–0.78, clearly outperforming ridge
> - This improvement over Part 1 is expected: the larger dataset gives the network more signal to learn the non-linear geography effects

In [ ]:
test_loss_r, test_mae_r = regression_model.evaluate(X_test_r_scaled, y_test_r, verbose=0)
pred_r = regression_model.predict(X_test_r_scaled, verbose=0).ravel()

nn_rmse = np.sqrt(mean_squared_error(y_test_r, pred_r))
nn_mae = mean_absolute_error(y_test_r, pred_r)
nn_r2 = r2_score(y_test_r, pred_r)

print("Neural network regression:")
print(f"RMSE: {nn_rmse:.3f}")
print(f"MAE:  {nn_mae:.3f}")
print(f"R^2:  {nn_r2:.3f}")

print("\nBaseline ridge regression:")
print(f"RMSE: {baseline_rmse:.3f}")
print(f"MAE:  {baseline_mae:.3f}")
print(f"R^2:  {baseline_r2:.3f}")

### 5.7 Observed vs. predicted plot

A scatter plot of observed vs. predicted values is a graphical check on model quality:

- **Perfect predictions** would fall exactly on the diagonal reference line.
- **Systematic bias** appears as a curve or tilt away from the diagonal.
- **Heteroscedasticity** appears as a funnel shape where predictions become less accurate at extreme values (very cheap or very expensive houses).

Notice whether the neural network's predictions are more tightly clustered around the diagonal than you would expect from the RMSE alone, or whether certain ranges of house values are systematically under- or over-predicted.

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_r, pred_r, alpha=0.3)
plt.xlabel("Observed median house value")
plt.ylabel("Predicted median house value")
plt.title("Regression neural network: observed vs predicted")
plt.grid(True)

lims = [min(y_test_r.min(), pred_r.min()), max(y_test_r.max(), pred_r.max())]
plt.plot(lims, lims)
plt.show()

### 5.8 Regression reflection with Sample answers

Answer these questions after trying several configurations.

**1. Did the neural network beat ridge regression?**

> **Sample answer**: With only 30 epochs and a small network (`[16]`), probably not as the NN is roughly equivalent to ridge regression. With a medium network (`[64, 32]`) and 100 epochs, the NN clearly outperforms ridge (lower RMSE, higher R²). This is different from the classification result, and the key reason is dataset size: ~15,000 training rows give the NN enough signal to learn the non-linear interactions between latitude, longitude, and house value that a linear model cannot capture.

**2. Did a larger network improve test RMSE?**

> **Sample answer**: Up to a point, yes. Moving from `[16]` to `[64, 32]` usually reduces RMSE because the larger network can model non-linear geography effects. Moving further to `[128, 64, 32]` gives diminishing returns and sometimes increases test RMSE if the model overfits. The validation loss curve is the clearest guide: stop adding capacity when validation loss stops improving.

**3. Did validation loss continue improving as training loss improved?**

> **Sample answer**: Initially yes; both curves fall together. After some number of epochs (varies by architecture), training loss continues to decline while validation loss levels off. This is the overfitting transition. The epoch at which validation loss is minimized is the ideal stopping point. Some frameworks provide `EarlyStopping` callbacks to automate this.

**4. Did dropout help or hurt?**

> **Sample answer**: With a small network (`[16]`) and 30 epochs, dropout has little effect because there is minimal overfitting to begin with. With a larger network or more epochs, `dropout_rate=0.2` tends to reduce the train/validation gap and produce a lower test RMSE. `dropout_rate=0.5` can be too aggressive for a small dataset and may cause underfitting (both training and validation loss remain high).

**5. Which configuration would you choose and why?**

> **Sample answer**: A reasonable choice is `hidden_layers=[64, 32]`, `learning_rate=0.001`, `epochs=100`, `batch_size=32`, `dropout_rate=0.2`. This configuration balances capacity (enough neurons to capture non-linearities) with regularisation (dropout prevents memorisation). It achieves RMSE around 0.55–0.65 and R² around 0.72–0.78, meaningfully better than ridge regression.

### 5.7 Compare many configurations automatically

The next cell runs several neural network configurations automatically.

This is useful for seeing that hyperparameter choices can matter, but also that bigger models do not always win.

The loop trains six regression models with different architectures, learning rates, and dropout rates, all for 20 epochs. Results are collected in a DataFrame and sorted by RMSE so you can compare them at a glance.

> **Tip**: Because models are trained with only 20 epochs, the differences here are smaller than you would see with longer training. Treat this as a quick screening, not a definitive comparison.

In [ ]:
configs = [
    {"hidden_layers": [8], "learning_rate": 0.001, "dropout_rate": 0.0},
    {"hidden_layers": [16], "learning_rate": 0.001, "dropout_rate": 0.0},
    {"hidden_layers": [64, 32], "learning_rate": 0.001, "dropout_rate": 0.0},
    {"hidden_layers": [128, 64, 32], "learning_rate": 0.001, "dropout_rate": 0.0},
    {"hidden_layers": [64, 32], "learning_rate": 0.01, "dropout_rate": 0.0},
    {"hidden_layers": [64, 32], "learning_rate": 0.001, "dropout_rate": 0.2},
]

results = []

for cfg in configs:
    tf.random.set_seed(427)
    model = build_regression_model(input_dim=X_train_r_scaled.shape[1], hidden_layers=cfg["hidden_layers"], activation="relu", learning_rate=cfg["learning_rate"], dropout_rate=cfg["dropout_rate"], l2_penalty=0.0)
    model.fit(X_train_r_scaled, y_train_r, validation_split=0.2, epochs=20, batch_size=32, verbose=0)
    preds = model.predict(X_test_r_scaled, verbose=0).ravel()
    results.append({
        "hidden_layers": str(cfg["hidden_layers"]),
        "learning_rate": cfg["learning_rate"],
        "dropout_rate": cfg["dropout_rate"],
        "rmse": np.sqrt(mean_squared_error(y_test_r, preds)),
        "mae": mean_absolute_error(y_test_r, preds),
        "r2": r2_score(y_test_r, preds)
    })

pd.DataFrame(results).sort_values("rmse")

## 6. Final discussions and Sample answers

Use your results to discuss the following claim:

> A neural network is not automatically better than a simpler model.

In your answer, refer to: **model complexity**, **sample size**, **overfitting**, **interpretability**, **training time**, **test-set performance**, and **the difference between tabular data and image/text data**.

---

**Sample answer**:

The experiments in this notebook confirm that a neural network is *not* automatically the best choice.

**Model complexity and sample size**: In Part 1 (569 samples, 30 features), logistic regression matched or outperformed every neural network we tried. The breast cancer dataset is small enough that the decision boundary is approximately linear, and a logistic regression captures that with zero tuning effort. A neural network in this setting has too many parameters relative to the data — even a `[16]`-neuron single-layer network has hundreds of parameters to fit from 456 training rows.

**Overfitting**: When we used a large architecture (`[128, 64, 32]`) or many epochs on the classification dataset, training accuracy approached 100% while test accuracy fell, a textbook overfitting case. On the larger California Housing dataset (Part 2, ~15,000 training rows), the same architecture caused much less overfitting because there was enough data to constrain the parameters. This shows that the risk of overfitting is jointly determined by model capacity *and* dataset size.

**Training time**: Even on free Colab CPUs, neural networks take seconds to minutes per run, and tuning requires many runs. Logistic regression and ridge regression train in milliseconds. When results are comparable, the simpler and faster model is the rational choice.

**Interpretability**: Logistic regression coefficients have direct interpretations (log-odds per unit change in a feature). A neural network is a black box; even small networks give no intuitive explanation of which features drive predictions or how. In a clinical setting like breast cancer diagnosis, interpretability can be as important as accuracy.

**Where neural networks win**: On the California Housing data, a sufficiently large and well-tuned network beat ridge regression by a meaningful margin. The reason is that the relationship between latitude/longitude and house value is *non-linear* — coastal geography creates price gradients that no linear model can represent. Neural networks with ReLU activations can approximate arbitrarily complex functions given enough data and capacity.

**Tabular vs. image/text data**: The advantage of neural networks is most pronounced for *unstructured* data. Images have spatial structure that convolutional layers exploit; text has sequential dependencies that transformers exploit. For clean tabular data with engineered features, linear models, gradient-boosted trees, and SVMs often perform just as well or better. Neural networks are most worth their complexity cost when the raw data requires the network to *learn* its own representations — something unnecessary when a domain expert has already constructed good features.

**Conclusion**: Choose your model based on the data characteristics, sample size, interpretability requirements, and available compute. Do not assume that more complexity is always better.